In [27]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("sci.mplstyle")

In [28]:
def tinh_he_so_backward(kappa, C, rho, N_x, N_time, L, t_max):
    h = L / (N_x - 1) # buoc luoi khong gian
    k = t_max / (N_time - 1) # buoc luoi thoi gian

    alpha = np.sqrt(kappa / (C * rho))

    eta = alpha**2 * k / h**2

    beta1 = eta / (1.0 + 2.0 * eta)
    beta2 = 1.0 / (1.0 + 2.0 * eta)

    return eta, beta1, beta2, h, k

In [29]:
def ham_ghi_file(file, dk_bai_toan): # dung de ghi thong tin quan trong ra file

    file.write(f"# {dk_bai_toan['mo_ta']}\n")
    file.write("# Backward Euler cho phuong trinh truyen nhiet 1D\n")

    file.write("#\n")

    file.write(f"# N_x    = {dk_bai_toan['N_x']}\n")
    file.write(f"# N_time = {dk_bai_toan['N_time']}\n")
    file.write(f"# L      = {dk_bai_toan['L']}\n")
    file.write(f"# t_max  = {dk_bai_toan['t_max']}\n")
    file.write(f"# h      = {dk_bai_toan['h']:.8e}\n")
    file.write(f"# k      = {dk_bai_toan['k']:.8e}\n")
    file.write(f"# eta    = {dk_bai_toan['eta']:.8e}\n")

    file.write("#\n")
    file.write(f"# {'j (t)':>8s} {'i (x)':>8s} {'t':>15s} {'x':>15s} {'u':>15s}\n")

def ham_luu_dulieu(filename, u, dk_bai_toan, skip_buoc_thoigian):
    N_x = dk_bai_toan["N_x"]
    N_time = dk_bai_toan["N_time"]
    h = dk_bai_toan["h"]
    k = dk_bai_toan["k"]

    with open(filename, "w", encoding="utf-8") as file:
        ham_ghi_file(file, dk_bai_toan)

        for j in range(N_time):
            if j % skip_buoc_thoigian != 0 and j != N_time - 1: # khi j khong chia het cho skip_buoc_thoigian thi bo qua data do va lay luon  data cuoi cung
                continue
            t = j * k

            for i in range(N_x):
                x = i * h
                file.write(f"  {j:8d} {i:8d} {t:15.6e} {x:15.6e} {u[i, j]:15.6e}\n")

            file.write("\n")

def ham_luu_hoitu(filename, bang_hoitu): # dung de luu so vong lap hoi tu

    with open(filename, "w", encoding="utf-8") as file:
        file.write("# Ket qua hoi tu cua Backward Euler\n")
        file.write(f"# {'j (t_step)':>15s} {'q (so vong lap)':>15s} {'max_err':>15s}\n")

        for j, q, max_err in bang_hoitu:
            file.write(f"  {j:15d} {q:15d} {max_err:15.6e}\n")

In [30]:
def ham_tinh_backward_euler(u,kappa,C,rho,L,t_max,N_max,tol,filename,mo_ta, skip_buoc_thoigian = 20):

    N_x = u.shape[0]
    N_time = u.shape[1]

    eta, beta1, beta2, h, k = tinh_he_so_backward(
        kappa, C, rho, N_x, N_time, L, t_max
    )

    bang_hoitu = []

    # Vong lap cho thoi gian
    for j in range(1, N_time):

        # Lay nghiem cua buoc truoc lam gia tri doan ban dau
        for i in range(1, N_x - 1):
            u[i, j] = u[i, j - 1]

        # Lap de giai so, voi moi buoc thoi gian thi phai lap de hoi tu tai buoc do
        for q in range(1, N_max + 1):

            u_old = u[:, j].copy()

            for i in range(1, N_x - 1):
                u[i, j] = (
                    beta1 * (u[i + 1, j] + u[i - 1, j])
                    + beta2 * u[i, j - 1]
                )

            max_err = np.max(np.abs(u[:, j] - u_old)) # sai so lon nhat tai moi buoc x (lay lam dai dien cho buoc thoi gian do)

            if max_err < tol:
                bang_hoitu.append((j, q, max_err)) # ghi lai so vong lap cho moi lan hoi tu

                if np.mod(j, 50) == 0: # in ra so vong lap cho moi lan hoi tu
                    print(
                        f"j = {j}, hoi tu sau q = {q} vong lap, "
                        f"max_err = {max_err:.3e}"
                    )

                break
        # Ep 2 dieu kien bien
        u[0, j] = u[0,0]
        u[N_x-1, j] = u[N_x-1, 0]

    dk_bai_toan = {
        "mo_ta": mo_ta,
        "N_x": N_x,
        "N_time": N_time,
        "L": L,
        "t_max": t_max,
        "h": h,
        "k": k,
        "eta": eta,
    }

    ham_luu_dulieu(filename + "_backward_result.txt", u, dk_bai_toan, skip_buoc_thoigian)

    ham_luu_hoitu(filename + "_backward_hoitu.txt", bang_hoitu)

    print("Da tinh xong Backward Euler")
    print(f"eta = {eta:.6e}")
    print(f"beta1 = {beta1:.6e}")
    print(f"beta2 = {beta2:.6e}")
    print(f"Da luu file: {filename + "_backward_result.txt"}")

    return u, dk_bai_toan

In [31]:
# =========================================================
# ba thanh tiep xuc
# =========================================================

def tao_dieu_kien_bien_bai4(
    N_time,
    N_x,
    l_bar,
    T_thanh_1,
    T_thanh_2,
    T_thanh_3,
    T_bien_trai,
    T_bien_phai,
):

    L = 3.0 * l_bar
    x = np.linspace(0.0, L, N_x)
    u = np.zeros((N_x, N_time), dtype=float)

    # Dieu kien dau:
    # Neu x <= l_bar thi diem do nam tren thanh trai
    # Neu x >  l_bar thi diem do nam tren thanh phai
    for i in range(N_x):
        if x[i] <= l_bar:
            u[i, 0] = T_thanh_1
        elif l_bar <=x [i] <= 2*l_bar:
            u[i, 0] = T_thanh_2
        else:
            u[i, 0] = T_thanh_3

    # Dieu kien bien trai
    for j in range(N_time):
        u[0, j] = T_bien_trai

    # Dieu kien bien phai
    for j in range(N_time):
        u[N_x - 1, j] = T_bien_phai

    # Tai t = 0, uu tien dieu kien bien tai hai dau
    u[0, 0] = T_bien_trai
    u[N_x - 1, 0] = T_bien_phai

    return u, x, L

N_time = 1500
N_x = 101

l_bar = 1
t_max = 1800.0

T_thanh_1 = 30.0
T_thanh_2 = 60.0
T_thanh_3 = 100.0


T_bien_trai = 0.0
T_bien_phai = 0.0

kappa = 210.0
C = 900.0
rho = 2700.0

tol  = 1e-6
N_max = 10000

u, x, L = tao_dieu_kien_bien_bai4(
    N_time=N_time,
    N_x=N_x,
    l_bar=l_bar,
    T_thanh_1=T_thanh_1,
    T_thanh_2=T_thanh_2,
    T_thanh_3=T_thanh_3,
    T_bien_trai=T_bien_trai,
    T_bien_phai=T_bien_phai,
)

u, info = ham_tinh_backward_euler(
    u=u,
    kappa=kappa,
    C=C,
    rho=rho,
    L=L,
    t_max=t_max,
    N_max = N_max,
    tol = tol, 
    filename="truyen_nhiet_3thanh",
    mo_ta="Bai toan truyen nhiet 1D: hai thanh tiep xuc",
    skip_buoc_thoigian = 1,
)

j = 50, hoi tu sau q = 7 vong lap, max_err = 5.079e-07
j = 100, hoi tu sau q = 7 vong lap, max_err = 2.601e-07
j = 150, hoi tu sau q = 7 vong lap, max_err = 1.743e-07
j = 200, hoi tu sau q = 7 vong lap, max_err = 1.313e-07
j = 250, hoi tu sau q = 7 vong lap, max_err = 1.055e-07
j = 300, hoi tu sau q = 6 vong lap, max_err = 8.587e-07
j = 350, hoi tu sau q = 6 vong lap, max_err = 7.442e-07
j = 400, hoi tu sau q = 6 vong lap, max_err = 6.621e-07
j = 450, hoi tu sau q = 6 vong lap, max_err = 6.002e-07
j = 500, hoi tu sau q = 6 vong lap, max_err = 5.504e-07
j = 550, hoi tu sau q = 6 vong lap, max_err = 5.090e-07
j = 600, hoi tu sau q = 6 vong lap, max_err = 4.750e-07
j = 650, hoi tu sau q = 6 vong lap, max_err = 4.445e-07
j = 700, hoi tu sau q = 6 vong lap, max_err = 4.183e-07
j = 750, hoi tu sau q = 6 vong lap, max_err = 3.941e-07
j = 800, hoi tu sau q = 6 vong lap, max_err = 3.731e-07
j = 850, hoi tu sau q = 6 vong lap, max_err = 3.536e-07
j = 900, hoi tu sau q = 6 vong lap, max_err = 3.3